In [ ]:
BITSTREAM_PATH = '../overlay/friscv.bit'
MEM_SIZE_B = 32 * 1024 * 1024
RESULT_OFFSET = 1280
RSICV_BASE = 0x8000_0000

In [ ]:
import os
import time
import ipywidgets as widgets
from pynq import allocate
from IPython.display import display
from debug_driver import DebugDriver

In [ ]:
if not os.path.exists(BITSTREAM_PATH):
    raise FileNotFoundError(f'Bitstream not found at {BITSTREAM_PATH}. Run \'make bitstream\'.')

# Create driver with dummy base address initially, will be updated after memory allocation
driver = DebugDriver(BITSTREAM_PATH, 0x00000000, start=False)

In [ ]:
# Allocate memory and zero
mem_size_w = MEM_SIZE_B // 4
mem = allocate(shape=(mem_size_w), dtype='u4')
dram_base = mem.physical_address
mem[:] = [0] * mem_size_w
mem.flush()

# Update driver with actual base address
driver.base_addr = dram_base
print("driver.base_addr (readback):", hex(driver.base_addr))
print("mem.phys", hex(mem.physical_address))

print(f"Memory allocated: {MEM_SIZE_B // (1024*1024)} MiB at physical address {hex(dram_base)}")
print(f"CPU view: {hex(RSICV_BASE)} - {hex(RSICV_BASE + MEM_SIZE_B - 1)}")

def get_result(mem, addr = RESULT_OFFSET):
    return mem[addr // 4]  # Memory is word-addressable, address is in bytes

def hex_dump(mem, from_cpu_addr, num_bytes=64):
    mem.invalidate()

    lines = []
    header = f"{'Address':<10} | {'00 01 02 03':<11}"
    lines.append(header)
    lines.append('-' * len(header))

    base_addr_offset = from_cpu_addr - RSICV_BASE
    idx_of_first = base_addr_offset // 4

    byte_list = []
    limit = min(num_bytes // 4, len(mem))
    for i in range(limit):
        val = mem[idx_of_first + i]
        byte_list.extend([val & 0xFF, (val >> 8) & 0xFF, (val >> 16) & 0xFF, (val >> 24) & 0xFF])

    for i in range(0, len(byte_list), 4):
        chunk = byte_list[i:i+4]
        hex_strs = [f'{b:02X}' for b in chunk]
        row_str = ' '.join(hex_strs)
        lines.append(f'{hex(base_addr_offset+i):<10} | {row_str}')

    return '\n'.join(lines)

In [ ]:
w_file_upload = widgets.FileUpload(
    accept='.bin',
    multiple=False,
    description='Load Binary',
    button_style='info'
)

btn_start = widgets.Button(description='Start', button_style='success')
btn_reset_start = widgets.Button(description='Reset+Start', button_style='warning')
btn_reset_wait = widgets.Button(description='Reset+Wait', button_style='warning')

w_result = widgets.Text(
    value='',
    description='Result:',
    disabled=True,
    layout={'width': '300px'}
)

w_result_addr = widgets.Text(
    value='',
    description='Address:',
    disabled=True,
    layout={'width': '350px'}
)

out_status = widgets.Output(layout={'border': '1px solid #444', 'padding': '10px'})

w_dump_addr = widgets.Text(
    value='0x80000000',
    description='Start addr:',
    layout={'width': '300px'}
)

w_dump_words = widgets.IntText(
    value=16,
    description='Dump words:',
    layout={'width': '200px'}
)

btn_dump = widgets.Button(description='Dump', button_style='info')

out_mem = widgets.Output(layout={'border': '1px solid #444', 'padding': '10px', 'font_family': 'monospace'})

def load_binary(change):
    """Load binary file into memory"""
    if len(w_file_upload.value) == 0:
        return
    
    uploaded_file = list(w_file_upload.value.values())[0]
    binary_data = uploaded_file['content']
    
    out_status.clear_output()
    with out_status:
        print(f"Loading {len(binary_data)} bytes into memory at base {hex(dram_base)}...")
        
        # Write binary data to memory (word by word)
        for i in range(0, len(binary_data), 4):
            word_bytes = binary_data[i:i+4]
            # Pad if less than 4 bytes
            while len(word_bytes) < 4:
                word_bytes += b'\x00'
            
            # Convert to little-endian word
            word = int.from_bytes(word_bytes, byteorder='little')
            mem[i // 4] = word
        
        mem.flush()
        print(f"Loaded successfully!")

def on_start_clicked(b):
    """Start the CPU"""
    driver.start()
    out_status.clear_output()
    with out_status:
        print("CPU started")
    update_result()

def on_reset_start_clicked(b):
    """Reset and start the CPU"""
    driver.reset()
    out_status.clear_output()
    with out_status:
        print("CPU reset and started")
    update_result()

def on_reset_wait_clicked(b):
    """Reset the CPU and keep it waiting"""
    driver.reset_stop()
    out_status.clear_output()
    with out_status:
        print("CPU reset and waiting")
    w_result.value = ''
    w_result_addr.value = ''

def update_result():
    """Update the result display"""
    result_val = get_result(mem, RESULT_OFFSET)
    result_phys_addr = dram_base + RESULT_OFFSET
    result_cpu_addr = RSICV_BASE + RESULT_OFFSET
    
    w_result.value = f"0x{result_val:08X} ({result_val})"
    w_result_addr.value = f"CPU: {hex(result_cpu_addr)}, Phys: {hex(result_phys_addr)}"

def on_dump_clicked(b):
    """Dump memory contents"""
    try:
        # Parse address (handle hex format)
        addr_str = w_dump_addr.value.strip()
        if addr_str.startswith('0x') or addr_str.startswith('0X'):
            start_addr = int(addr_str, 16)
        else:
            start_addr = int(addr_str)
        
        num_words = w_dump_words.value
        num_bytes = num_words * 4
        
        out_mem.clear_output()
        with out_mem:
            print(f'Memory Base (Physical): {hex(mem.physical_address)}')
            print(f'Dumping from CPU address {hex(start_addr)} ({num_words} words, {num_bytes} bytes)\n')
            print(hex_dump(mem, start_addr, num_bytes=num_bytes))
    
    except ValueError as e:
        out_mem.clear_output()
        with out_mem:
            print(f"Error: Invalid address format. Use hex (0x80000000) or decimal.")

w_file_upload.observe(load_binary, names='value')
btn_start.on_click(on_start_clicked)
btn_reset_start.on_click(on_reset_start_clicked)
btn_reset_wait.on_click(on_reset_wait_clicked)
btn_dump.on_click(on_dump_clicked)

ui = widgets.VBox([
    widgets.HTML("<h2>Program Control</h2>"),
    w_file_upload,
    widgets.HBox([btn_start, btn_reset_start, btn_reset_wait]),
    w_result,
    w_result_addr,
    out_status,
    widgets.HTML("<hr><h2>Memory Dump</h2>"),
    widgets.HBox([w_dump_addr, w_dump_words]),
    btn_dump,
    out_mem
], layout={'padding': '10px'})

In [ ]:
display(ui)